# 04b Validation V2: tubing method and Scafidi et al. (2023)

Run once per change to `tubing.py`; record results in `docs/validation_log.md`.

1. **V2.3:** average temperature and Z method (eq. 2) against RK4 integration (eq. 3) → printed table
2. **V2.1/V2.2:** Rough against Scafidi et al. (2023) → `04b_scafidi_comparison.csv` (main text) and `04b_scafidi_attribution.csv` (appendix)

Their code is rebuilt in `validation/scafidi2023.py`, which also holds their published Rough results (`PUBLISHED_ROUGH_30`, from their supplementary file si_002).

In [ ]:
%load_ext autoreload
%autoreload 2
import subprocess, hashlib, datetime, pathlib
root = pathlib.Path.cwd().parent
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=root).stdout.strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, cwd=root).stdout.strip() != ""
inp = hashlib.sha256((root / "data" / "inputs.csv").read_bytes()).hexdigest()[:10]
wls = hashlib.sha256((root / "data" / "wells.csv").read_bytes()).hexdigest()[:10]
print(f"Run {datetime.datetime.now():%Y-%m-%d %H:%M} | commit {commit}{' (UNCOMMITTED CHANGES)' if dirty else ''} | inputs.csv {inp} | wells.csv {wls}")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from h2_bottleneck import inputs, props
from h2_bottleneck import well as wl
from h2_bottleneck import tubing as tb

FIG, TAB = root / "results" / "figures", root / "results" / "tables"
GASES = ("Hydrogen", "Methane")
COL = {"Hydrogen": "tab:blue", "Methane": "tab:orange"}
DAY, MSM3, BAR, IN = 86400.0, 1e6, 1e5, 0.0254

x = inputs.load()
f = wl.Formation.from_inputs(x)
wells = [wl.Well.from_row(r) for _, r in inputs.load_wells().iterrows()]
tub = tb.Tubing.from_inputs(x, f)
tubs = {w.name: tub for w in wells}                 # A15: same tubing in every well
m = {g: wl.Pseudopressure(f.T, g) for g in GASES}
print(f"Tubing: {tub.tvd:.0f} m vertical, ID {tub.d / IN:.1f} in, T_wh {tub.T_wh - 273.15:.0f} °C, T_res {tub.T_bh - 273.15:.1f} °C")

import sys
sys.path.insert(0, str(root))
from validation import scafidi2023 as sc

## 1. V2.3: average method against RK4

WHP 150 bar; error = average method − RK4, in bar and as % of the static gas column. Tolerance in `test_tubing.py`: 4 % of the column.

In [ ]:
from dataclasses import replace
rows = []
for d_in in (2.0, 7.0):
    t_d = replace(tub, d=d_in * IN)
    for g in GASES:
        col = tb.pwf_from_pwh(150e5, 0.0, t_d, g) - 150e5
        for q_d in (-0.05 * d_in**2, 0.0, 0.05 * d_in**2):            # ~0.2 (2 in) to ~2.5 (7 in) million Sm3/d
            q = q_d * MSM3 / DAY
            err = tb.pwf_from_pwh(150e5, q, t_d, g) - tb.pwf_rk4(150e5, q, t_d, g)
            rows.append(dict(tubing_in=d_in, gas=g, q_MSm3d=round(q_d, 2), error_bar=err / BAR, error_pct_of_column=100 * err / col))
pd.DataFrame(rows).pivot_table(index=["tubing_in", "q_MSm3d"], columns="gas", values=["error_bar", "error_pct_of_column"]).round(2)

## 2. V2.1/V2.2: Rough against Scafidi et al. (2023)

Their conditions: 30 wells, p_res 4533 psia, WHP 50 % of p_res, vertical 9000 ft, 2–8 in, both gases. Cushion gas requirement (CGR, %) from their eqs. 16–18.

| Column | Meaning |
|---|---|
| published | their supplementary results (si_002), held in `sc.PUBLISHED_ROUGH_30`; Table 4 quotes the CH4 2 and 8 in values |
| their_code | `sc.run`: their code rebuilt, as published (should match `published`) |
| their_code_corrected | `sc.run` with the four departures from the printed equations corrected |
| this_model_generic | this model with their generic well (`sc.this_model`): should match `their_code_corrected` |
| this_model_29_wells | this model with the 29 Heinemann wells (mean rate × 30) |

In [ ]:
rows = []
t0 = time.time()
for g in GASES:
    for d in (2, 4, 6, 8):
        rows.append(dict(gas=g, tubing_in=d,
                         published=sc.PUBLISHED_ROUGH_30[(sc.GAS[g], d)],
                         their_code=sc.run("Rough", sc.GAS[g], d, 30)["cgr"],
                         their_code_corrected=sc.run("Rough", sc.GAS[g], d, 30, **sc.CORRECTED)["cgr"],
                         this_model_generic=sc.this_model(d, g, x, f, tub, m[g])["cgr"],
                         this_model_29_wells=sc.this_model(d, g, x, f, tub, m[g], wells=wells)["cgr"]))
comp = pd.DataFrame(rows)
comp.to_csv(TAB / "04b_scafidi_comparison.csv", index=False)
print(f"Comparison in {time.time() - t0:.0f} s; reported actual CGR for Rough: 64.2 %")
comp.round(1)

**Attribution (appendix):** each departure corrected on its own, Rough CH4, 30 wells. At 2 in the well is tubing-limited (friction issues dominate); at 8 in it is inflow-limited (eq. 3 dominates). Their other CH4 validation cases (8 in) are printed below.

In [ ]:
rows = [dict(case="their code as published", **{f"{d}_in": sc.run("Rough", "CH4", d, 30)["cgr"] for d in (2, 8)})]
for key, label in sc.ISSUES.items():
    rows.append(dict(case=f"fix only: {label}", **{f"{d}_in": sc.run("Rough", "CH4", d, 30, **{key: False})["cgr"] for d in (2, 8)}))
rows.append(dict(case="all four fixed", **{f"{d}_in": sc.run("Rough", "CH4", d, 30, **sc.CORRECTED)["cgr"] for d in (2, 8)}))
attr = pd.DataFrame(rows)
attr.to_csv(TAB / "04b_scafidi_attribution.csv", index=False)
for field, n, actual in (("Grijpskerk", 1, 88.5), ("Hatfield Moors", 2, 66.7)):
    print(f"{field} ({n} well, 8 in): their code {sc.run(field, 'CH4', 8, n)['cgr']:.1f} %, "
          f"corrected {sc.run(field, 'CH4', 8, n, **sc.CORRECTED)['cgr']:.1f} %, reported actual {actual} %")
attr.round(1)